In [ ]:
pip install merlin-vlm

In [ ]:
pip install datasets huggingface_hub nibabel numpy torch torchvision

In [ ]:
import os, json, numpy as np, nibabel as nib, torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

PROJECT_ROOT = os.path.expanduser("~/Documents/GitHub/fyp-3d-ct/models/merlin")
DATA_ROOT    = os.path.join(PROJECT_ROOT, "data")
VOLUMES_DIR  = os.path.join(DATA_ROOT, "data_volumes")
REXCT_DIR    = os.path.join(DATA_ROOT, "rexgrounding-ct")
DATASET_JSON = os.path.join(REXCT_DIR, "dataset.json")

os.makedirs("results", exist_ok=True)
print("Paths set.")

In [ ]:
ct_index = {}
for root, dirs, files in os.walk(VOLUMES_DIR):
    for f in files:
        if f.endswith(".nii.gz"):
            ct_index[f] = os.path.join(root, f)

print(f"CT files found: {len(ct_index)}")
for name in ct_index:
    print(f"  {name}")

In [ ]:
with open(DATASET_JSON) as f:
    metadata = json.load(f)

samples = metadata if isinstance(metadata, list) else list(metadata.values())[0]
matched = [s for s in samples if s["name"] in ct_index]

print(f"Matched {len(matched)} samples ready to process")

In [ ]:
def load_and_preprocess_ct(nii_path):
    img    = nib.load(nii_path)
    volume = img.get_fdata()
    volume = np.clip(volume, -1000, 400)
    volume = (volume + 1000) / 1400.0
    volume = np.nan_to_num(volume, nan=0.0)
    tensor = torch.tensor(volume, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    return tensor, img.affine, volume

In [ ]:
from merlin import Merlin

model = Merlin(ImageEmbedding=True)
model.eval()
print("Merlin loaded!")

In [ ]:
import urllib.request

VALID_CATEGORIES = ["Lung Nodule", "Lung opacity", "Consolidation", "Atelectasis"]

def classify_finding(finding_text: str) -> str:
    """Send finding text to Claude API and get one of 5 fixed categories."""
    
    prompt = f"""Analyze the provided input and identify the lung condition. You must strictly limit your output to one of the following exact categories: 'Lung Nodule', 'Lung opacity', 'Consolidation', or 'Atelectasis'. If any other disease, abnormality, or condition is detected that is not on this list, classify it strictly as 'Others'. Output only the category name, without any extra text or explanations.

Input: {finding_text}"""

    payload = json.dumps({
        "model": "claude-sonnet-4-20250514",
        "max_tokens": 20,
        "messages": [{"role": "user", "content": prompt}]
    }).encode("utf-8")

    req = urllib.request.Request(
        "https://api.anthropic.com/v1/messages",
        data    = payload,
        headers = {
            "Content-Type"      : "application/json",
            "anthropic-version" : "2023-06-01"
        },
        method = "POST"
    )

    try:
        with urllib.request.urlopen(req) as resp:
            data     = json.loads(resp.read().decode())
            category = data["content"][0]["text"].strip()
            # Validate — if model returns something unexpected, force Others
            if category not in VALID_CATEGORIES:
                category = "Others"
            return category
    except Exception as e:
        print(f"    API error: {e}")
        return "Others"


def classify_all_findings(findings: dict) -> dict:
    """Classify every finding in a sample. findings = {"0": "text", "1": "text"}"""
    classified = {}
    for idx, text in findings.items():
        category = classify_finding(text)
        classified[idx] = {
            "text"    : text,
            "category": category
        }
        print(f"    Finding {int(idx)+1}: {category}")
        print(f"      → {text[:70]}")
    return classified

In [ ]:
def build_localization_mask(volume_norm, emb_np):
    H, W, D      = volume_norm.shape
    emb_scalar   = float(np.mean(np.abs(emb_np.flatten())))
    act_volume   = np.zeros((H, W, D), dtype=np.float32)

    for d in range(D):
        act_volume[:, :, d] = volume_norm[:, :, d] * emb_scalar

    threshold   = np.percentile(act_volume, 95)
    binary_mask = (act_volume >= threshold).astype(np.uint8)
    return binary_mask, threshold

In [ ]:
def visualize_ct_with_mask(name, ct_index, out_dir="results"):
    stem      = name.replace(".nii.gz", "")
    out       = os.path.join(out_dir, stem)
    mask_path = os.path.join(out, "localization_mask.nii.gz")

    ct_vol   = nib.load(ct_index[name]).get_fdata()
    mask_vol = nib.load(mask_path).get_fdata()

    ct_display = np.clip(ct_vol, -1000, 400)
    ct_display = (ct_display + 1000) / 1400.0

    H, W, D = ct_vol.shape

    slices_with_mask = [d for d in range(D) if mask_vol[:, :, d].sum() > 0]
    if not slices_with_mask:
        slices_with_mask = list(range(0, D, max(1, D // 6)))
    step          = max(1, len(slices_with_mask) // 6)
    chosen_slices = slices_with_mask[::step][:6]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.patch.set_facecolor("#0e0e0e")
    fig.suptitle(f"CT + Localization Mask\n{name}",
                 color="white", fontsize=13, fontweight="bold", y=0.98)

    for ax, d in zip(axes.flat, chosen_slices):
        ct_slice   = ct_display[:, :, d].T
        mask_slice = mask_vol[:, :, d].T
        ax.imshow(ct_slice, cmap="gray", origin="lower", vmin=0, vmax=1)
        masked = np.ma.masked_where(mask_slice == 0, mask_slice)
        ax.imshow(masked, cmap="Reds", origin="lower", alpha=0.45, vmin=0, vmax=1)
        ax.set_title(f"Slice {d}/{D}", color="white", fontsize=9)
        ax.axis("off")

    ct_patch   = mpatches.Patch(color="gray", label="CT scan")
    mask_patch = mpatches.Patch(color="red",  alpha=0.6, label="Localization mask")
    fig.legend(handles=[ct_patch, mask_patch],
               loc="lower center", ncol=2,
               facecolor="#1a1a1a", edgecolor="white",
               labelcolor="white", fontsize=10,
               bbox_to_anchor=(0.5, 0.01))

    plt.tight_layout(rect=[0, 0.05, 1, 0.96])

    img_path = os.path.join(out, "ct_mask_overlay.png")
    plt.savefig(img_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()
    print(f"  ✓ Visualization saved → {img_path}")

In [ ]:
def run_full_pipeline(sample, model, ct_index, out_dir="results"):
    name     = sample["name"]
    findings = sample["findings"]
    shape    = sample["shape"]
    cats     = sample.get("categories", {})
    pixels   = sample.get("pixels", {})
    protocol = sample.get("protocol", "unknown")

    print(f"\n{'='*55}")
    print(f"Processing : {name}")
    print(f"Protocol   : {protocol}  |  Findings: {len(findings)}")
    print(f"{'='*55}")

    # Load CT
    ct_tensor, affine, volume_norm = load_and_preprocess_ct(ct_index[name])
    print(f"  CT loaded — shape: {ct_tensor.shape}")

    # Merlin embedding
    with torch.no_grad():
        embedding = model(ct_tensor)
    emb_np = embedding.cpu().numpy()
    print(f"  Embedding — shape: {emb_np.shape}  norm: {np.linalg.norm(emb_np):.4f}")

    # Classify findings via Claude API
    print(f"  Classifying {len(findings)} finding(s)...")
    classified = classify_all_findings(findings)

    # Build localization mask
    binary_mask, threshold = build_localization_mask(volume_norm, emb_np)
    print(f"  Mask — active voxels: {binary_mask.sum()}  threshold: {threshold:.4f}")

    # Build report
    finding_lines = []
    for idx, info in classified.items():
        px  = pixels.get(idx, "N/A")
        cat = cats.get(idx, "N/A")
        finding_lines.append(
            f"  Finding {int(idx)+1}\n"
            f"    Classification : {info['category']}\n"
            f"    Category code  : {cat}\n"
            f"    Pixel count    : {px}\n"
            f"    Text           : {info['text']}"
        )

    report = f"""
MERLIN CT ANALYSIS REPORT
{'='*55}
File       : {name}
Protocol   : {protocol}
Shape      : {shape[0]} x {shape[1]} x {shape[2]} voxels

EMBEDDING
  Shape    : {emb_np.shape}
  Norm     : {np.linalg.norm(emb_np):.4f}
  Mean     : {np.mean(emb_np):.4f}
  Std      : {np.std(emb_np):.4f}

FINDINGS ({len(findings)} total)
{chr(10).join(finding_lines)}

LOCALIZATION MASK
  Active voxels : {binary_mask.sum()}
  Threshold     : {threshold:.4f}  (top 5% activation)
  Coverage      : {100*binary_mask.sum()/volume_norm.size:.2f}% of volume
{'='*55}
"""
    print(report)

    # ── Save outputs ──────────────────────────────────────────
    stem = name.replace(".nii.gz", "")
    out  = os.path.join(out_dir, stem)
    os.makedirs(out, exist_ok=True)

    # 1. Report
    rpt_path = os.path.join(out, "report.txt")
    with open(rpt_path, "w") as f:
        f.write(report)
    print(f"  ✓ report.txt              → {rpt_path}")

    # 2. Embeddings
    emb_path = os.path.join(out, "embedding.npz")
    np.savez(emb_path, embedding=emb_np)
    print(f"  ✓ embedding.npz           → {emb_path}")

    # 3. Localization mask
    mask_nii  = nib.Nifti1Image(binary_mask, affine)
    mask_path = os.path.join(out, "localization_mask.nii.gz")
    nib.save(mask_nii, mask_path)
    print(f"  ✓ localization_mask.nii.gz → {mask_path}")

    # 4. Findings JSON
    findings_out = {
        "name"              : name,
        "protocol"          : protocol,
        "shape"             : shape,
        "classified_findings": classified,
        "categories"        : cats,
        "pixels"            : pixels,
        "embedding_norm"    : float(np.linalg.norm(emb_np)),
        "mask_active_voxels": int(binary_mask.sum())
    }
    findings_path = os.path.join(out, "findings.json")
    with open(findings_path, "w") as f:
        json.dump(findings_out, f, indent=2)
    print(f"  ✓ findings.json           → {findings_path}")

    # 5. Visualization
    visualize_ct_with_mask(name, ct_index, out_dir)

    return {"name": name, "report": report,
            "embedding": emb_np, "mask": binary_mask,
            "findings": findings_out}

In [ ]:
all_results = []

for sample in matched[:2]:
    result = run_full_pipeline(sample, model, ct_index)
    all_results.append(result)

print(f"\n{'='*55}")
print(f"DONE — {len(all_results)} volumes processed")
print(f"\nOutput structure:")
for r in all_results:
    stem = r["name"].replace(".nii.gz", "")
    print(f"\n  results/{stem}/")
    print(f"    ├── report.txt")
    print(f"    ├── embedding.npz")
    print(f"    ├── localization_mask.nii.gz")
    print(f"    ├── findings.json")
    print(f"    └── ct_mask_overlay.png")

In [ ]:
# step = 1
import os
import gc
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("GPU cleaned")

#step = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

#step = 3
import os

PROJECT_ROOT = os.path.expanduser("/home/chest_ct/code")

DATA_ROOT = os.path.join(PROJECT_ROOT, "data")

VOLUMES_DIR = os.path.join(DATA_ROOT, "segmentations/segmentations")

REXCT_DIR = os.path.join(DATA_ROOT, "rexgrounding-ct")

DATASET_JSON = os.path.join(REXCT_DIR, "dataset.json")

RESULTS_DIR = os.path.join(PROJECT_ROOT, "models/merlin/results_merlin")

os.makedirs(RESULTS_DIR, exist_ok=True)

# step = 4
import json

with open(DATASET_JSON, "r") as f:
    metadata = json.load(f)

samples = metadata if isinstance(metadata, list) else list(metadata.values())[0]

print("Total samples:", len(samples))

# step = 5
ct_index = {}

import os

for root, dirs, files in os.walk(VOLUMES_DIR):
    for f in files:
        if f.endswith(".nii.gz"):
            ct_index[f] = os.path.join(root, f)

print("CT files:", len(ct_index))

# step = 6
matched = [s for s in samples if s["name"] in ct_index]

print("Matched:", len(matched))

# step = 7
import numpy as np
from scipy.ndimage import zoom

def safe_ct(ct):
    ct = np.asarray(ct)

    while ct.ndim > 3:
        ct = np.squeeze(ct)

    if ct.ndim != 3:
        raise ValueError(f"Invalid CT shape: {ct.shape}")

    return ct


def resize_ct(ct, target=(96,96,96)):
    factors = [
        target[i] / ct.shape[i]
        for i in range(3)
    ]
    return zoom(ct, factors, order=1)

# step = 8
from merlin import Merlin

print("Loading embedding model...")

model_embed = Merlin(ImageEmbedding=True).to(DEVICE).eval()

print("Embedding model loaded")

# step = 9
import nibabel as nib
from rich.console import Console

console = Console()

def process_embedding(sample):

    name = sample["name"]
    ct_path = ct_index[name]

    try:
        console.print(f"\nProcessing: {name}")

        # LOAD CT
        ct_raw = nib.load(ct_path)
        ct = ct_raw.get_fdata()
        affine = ct_raw.affine

        # SAFE
        ct = safe_ct(ct)

        # NORMALIZE
        ct = np.clip(ct, -1000, 400)
        ct = (ct + 1000) / 1400.0
        ct = np.nan_to_num(ct)

        # RESIZE
        ct = resize_ct(ct)

        # TO TENSOR
        ct_tensor = torch.tensor(ct, dtype=torch.float32)
        ct_tensor = ct_tensor.unsqueeze(0).unsqueeze(0).to(DEVICE)

        # EMBEDDING
        with torch.inference_mode(), torch.cuda.amp.autocast():
            emb = model_embed(ct_tensor)

        emb_np = emb.detach().cpu().numpy()

        del ct_tensor, emb
        torch.cuda.empty_cache()
        gc.collect()

        return {
            "name": name,
            "embedding": emb_np,
            "affine": affine,
            "ct_path": ct_path
        }

    except Exception as e:
        print("ERROR:", e)
        return None
    
# step = 10
results = []
errors = []

print("Running embedding phase...")

for i, sample in enumerate(matched):

    out = process_embedding(sample)

    if out is None:
        errors.append(sample["name"])
    else:
        results.append(out)

    print(f"[{i+1}/{len(matched)}] success={len(results)} errors={len(errors)}")

print("Embedding DONE")

# step = 11
del model_embed
gc.collect()

if torch.cuda.is_available()
    torch.cuda.empty_cache()

print("Embedding model removed from GPU")

# step = 12
from merlin import Merlin

print("Loading report model...")

model_report = Merlin(RadiologyReport=True).to(DEVICE).eval()

print("Report model loaded")

# step = 13
def generate_report(sample):

    try:
        ct_path = sample["ct_path"]

        from merlin.data import DataLoader

        loader = DataLoader(
            datalist=[{"image": ct_path, "text": ""}],
            cache_dir="./cache",
            batchsize=1,
            shuffle=False,
            num_workers=0
        )

        with torch.inference_mode(), torch.cuda.amp.autocast():

            for batch in loader:
                rep = model_report(batch["image"].to(DEVICE))
                return str(rep)

    except Exception as e:
        print("Report error:", e)
        return None
    
# step = 14
for i, r in enumerate(results):

report = generate_report(r)
results[i]["report"] = report

print(f"[{i+1}/{len(results)}] report done")

# step = 15
import json
import os

out_file = os.path.join(RESULTS_DIR, "final_results.json")

with open(out_file, "w") as f:
    json.dump(results, f, indent=2)

print("Saved:", out_file)


